# Rebound State Factor Target Contract Handoff v1

Purpose: bridge the factor-mining result into the model-selection frontier without training or promoting an actor.

Source factor card: `runtime/factor/card/rebound_state_factor_mining_v1`.

Source frontier card: `state_head_stable_rebound_label_screen_v1`.

Decision rule: if the factor card found a target-contract candidate but current fold-native target lift is still zero, mark this as target revision ready. The next step is to build a rebound-state target contract and rerun label stability before any state-head training.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate_root in (ROOT, *ROOT.parents):
    if (candidate_root / "src" / "soxl_quant").exists():
        ROOT = candidate_root
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT = "rebound_state_factor_target_contract_handoff_v1"
CARD_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
FACTOR_CARD = ROOT / "runtime" / "factor" / "card" / "rebound_state_factor_mining_v1"
SOURCE_FRONTIER = "state_head_stable_rebound_label_screen_v1"
CARD_DIR.mkdir(parents=True, exist_ok=True)

factor_summary = json.loads((FACTOR_CARD / "summary.json").read_text(encoding="utf-8"))["summary"]
factor_scores = pd.read_csv(FACTOR_CARD / "factor_candidate_scores.csv")
selected_rows = pd.read_csv(FACTOR_CARD / "selected_factor_rows.csv")
selected = factor_scores.iloc[0].to_dict()
print(f"factor_decision={factor_summary['decision']} selected={factor_summary['selected_candidate']}")

factor_decision=rebound_state_factor_candidate_found_target_contract_needed_no_training selected=spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75


In [2]:
metrics = {
    "selected_candidate": factor_summary["selected_candidate"],
    "selected_positive_rows": factor_summary["selected_positive_rows"],
    "selected_unique_positive_dates": factor_summary["selected_unique_positive_dates"],
    "selected_positive_years": factor_summary["selected_positive_years"],
    "selected_forward_return_5_lift": factor_summary["selected_forward_return_5_lift"],
    "selected_forward_return_10_lift": factor_summary["selected_forward_return_10_lift"],
    "selected_full_rows": factor_summary["selected_full_rows"],
    "selected_full_years": factor_summary["selected_full_years"],
    "selected_hold_5_max_drawdown": factor_summary["selected_hold_5_max_drawdown"],
    "selected_hold_5_cap035_max_drawdown": factor_summary["selected_hold_5_cap035_max_drawdown"],
    "selected_target_lift_mean": factor_summary["selected_target_lift_mean"],
    "target_contract_handoff_gate_candidates": factor_summary["target_contract_handoff_gate_candidates"],
    "direct_current_target_gate_candidates": factor_summary["direct_current_target_gate_candidates"],
}

next_action = "build_rebound_state_target_contract_from_spread_confirmed_pullback_then_rerun_state_head_label_stability"
decision = "rebound_state_target_contract_ready_no_training"
summary = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "promotion_allowed": False,
    "training_allowed": False,
    "target_contract_gate": False,
    "next_training_allowed_after_target_revision": True,
    "next_mechanism": next_action,
    "source_experiment": SOURCE_FRONTIER,
    "source_factor_experiment": "rebound_state_factor_mining_v1",
    "direct_comparison_baseline": SOURCE_FRONTIER,
    "reference_factor_baseline": "stable_rebound_label_discovery_v1",
    "primary_failure": "current_fold_native_target_lift_is_zero_for_selected_factor",
    **metrics,
}
selection = {
    **summary,
    "factor_card": "runtime/factor/card/rebound_state_factor_mining_v1/rebound_state_factor_mining_v1.ipynb",
    "factor_summary": "runtime/factor/card/rebound_state_factor_mining_v1/summary.json",
    "selected_factor_rows": "runtime/factor/card/rebound_state_factor_mining_v1/selected_factor_rows.csv",
    "handoff_note": "This card intentionally stops before training. Build a new target contract from the selected factor, rerun duplicate-date label stability, then only train if walk-forward gates are explicit.",
}

handoff_metrics = pd.DataFrame([metrics])
handoff_metrics.to_csv(CARD_DIR / "handoff_metrics.csv", index=False)
(CARD_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps({"summary": summary}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
summary

{'experiment': 'rebound_state_factor_target_contract_handoff_v1',
 'decision': 'rebound_state_target_contract_ready_no_training',
 'promotion_allowed': False,
 'training_allowed': False,
 'target_contract_gate': False,
 'next_training_allowed_after_target_revision': True,
 'next_mechanism': 'build_rebound_state_target_contract_from_spread_confirmed_pullback_then_rerun_state_head_label_stability',
 'source_experiment': 'state_head_stable_rebound_label_screen_v1',
 'source_factor_experiment': 'rebound_state_factor_mining_v1',
 'direct_comparison_baseline': 'state_head_stable_rebound_label_screen_v1',
 'reference_factor_baseline': 'stable_rebound_label_discovery_v1',
 'primary_failure': 'current_fold_native_target_lift_is_zero_for_selected_factor',
 'selected_candidate': 'spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75',
 'selected_positive_rows': 20,
 'selected_unique_positive_dates': 11,
 'selected_positive_years': 3,
 'selected_forward_return_5_lift': 0.11752518859252593,
 'sel

In [3]:
conclusion = f"""
## Conclusion

Decision: `{summary['decision']}`.

Selected factor: `{summary['selected_candidate']}`.

Evidence: target rows `{summary['selected_positive_rows']}`, unique dates `{summary['selected_unique_positive_dates']}`, target years `{summary['selected_positive_years']}`, 5d lift `{summary['selected_forward_return_5_lift']:.4f}`, 10d lift `{summary['selected_forward_return_10_lift']:.4f}`, full-history rows `{summary['selected_full_rows']}`, full-history years `{summary['selected_full_years']}`, 5d hold MaxDD `{summary['selected_hold_5_max_drawdown']:.4f}`.

Blocker: current fold-native target lift is `{summary['selected_target_lift_mean']:.4f}`, so this is target revision readiness, not state-head training permission.

Next action: `{summary['next_mechanism']}`.
""".strip()
display(Markdown(conclusion))

## Conclusion

Decision: `rebound_state_target_contract_ready_no_training`.

Selected factor: `spread_confirmed_pullback__mom5_leq0.20_ratio21_geq0.75`.

Evidence: target rows `20`, unique dates `11`, target years `3`, 5d lift `0.1175`, 10d lift `0.0517`, full-history rows `34`, full-history years `13`, 5d hold MaxDD `-0.1590`.

Blocker: current fold-native target lift is `0.0000`, so this is target revision readiness, not state-head training permission.

Next action: `build_rebound_state_target_contract_from_spread_confirmed_pullback_then_rerun_state_head_label_stability`.